In [1]:
# Basic Import
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt 
import seaborn as sns
# Modelling
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor,AdaBoostRegressor
from sklearn.svm import SVR
from sklearn.linear_model import LinearRegression, Ridge,Lasso
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.model_selection import RandomizedSearchCV
from catboost import CatBoostRegressor
from xgboost import XGBRegressor
import warnings


from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import VotingClassifier

In [2]:
df = pd.read_csv("../data/raw/diabetes/diabetes_raw.csv")

In [3]:
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72.0,35,169.5,33.6,0.627,50,1
1,1,85,66.0,29,102.5,26.6,0.351,31,0
2,8,183,64.0,32,169.5,23.3,0.672,32,1
3,1,89,66.0,23,94.0,28.1,0.167,21,0
4,0,137,40.0,35,168.0,43.1,2.288,33,1


In [4]:
X = df.drop("Outcome", axis=1)
y = df["Outcome"]

In [5]:
## Train Test Split
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [6]:
X_train

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age
60,2,84,70.0,27,102.5,30.1,0.304,21
618,9,112,82.0,24,169.5,28.2,1.282,50
346,1,139,46.0,19,83.0,28.7,0.654,22
294,0,161,50.0,27,102.5,21.9,0.254,65
231,6,134,80.0,37,370.0,46.2,0.238,46
...,...,...,...,...,...,...,...,...
71,5,139,64.0,35,140.0,28.6,0.411,26
106,1,96,122.0,27,102.5,22.4,0.207,27
270,10,101,86.0,37,169.5,45.6,1.136,38
435,0,141,74.5,32,169.5,42.4,0.205,29


In [7]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [8]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
## Now next code 
def evaluate_model(true, predicted):
    accuracy = accuracy_score(true, predicted)
    precision = precision_score(true, predicted, zero_division=0)
    recall = recall_score(true, predicted, zero_division=0)
    f1 = f1_score(true, predicted, zero_division=0)

    return accuracy, precision, recall, f1

In [9]:
models = {
    "Random Forest": RandomForestClassifier(random_state=42),
    "SVM": SVC(),
    "XGBoost": XGBClassifier(random_state=42, eval_metric="logloss"),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "KNN": KNeighborsClassifier(),
    "Naive Bayes": GaussianNB(),
    "AdaBoost": AdaBoostClassifier(random_state=42),
    "Decision Tree": DecisionTreeClassifier(random_state=42),

    "Hard Voting": VotingClassifier(
        estimators=[
            ("lr", LogisticRegression(max_iter=1000)),
            ("rf", RandomForestClassifier(random_state=42)),
            ("dt", DecisionTreeClassifier(random_state=42))
        ],
        voting="hard"
    ),

    "Soft Voting": VotingClassifier(
        estimators=[
            ("lr", LogisticRegression(max_iter=1000)),
            ("rf", RandomForestClassifier(random_state=42)),
            ("dt", DecisionTreeClassifier(random_state=42))
        ],
        voting="soft"
    )
}

In [10]:
model_list = []
accuracy_list = []
precision_list = []
recall_list = []
f1_list = []

In [11]:
model_list = []

train_accuracy_list = []
train_precision_list = []
train_recall_list = []
train_f1_list = []

accuracy_list = []
precision_list = []
recall_list = []
f1_list = []

In [12]:
for model_name, model in models.items():

    # Train model
    model.fit(X_train_scaled, y_train)

    # Predictions
    y_train_pred = model.predict(X_train_scaled)
    y_test_pred = model.predict(X_test_scaled)

    # Training metrics
    train_accuracy, train_precision, train_recall, train_f1 = evaluate_model(
        y_train,
        y_train_pred
    )

    # Testing metrics
    test_accuracy, test_precision, test_recall, test_f1 = evaluate_model(
        y_test,
        y_test_pred
    )

    # Store model name
    model_list.append(model_name)

    # Store Training metrics
    train_accuracy_list.append(train_accuracy)
    train_precision_list.append(train_precision)
    train_recall_list.append(train_recall)
    train_f1_list.append(train_f1)

    # Store Testing metrics
    accuracy_list.append(test_accuracy)
    precision_list.append(test_precision)
    recall_list.append(test_recall)
    f1_list.append(test_f1)

In [13]:
results = pd.DataFrame(
    list(zip(
        model_list,

        train_accuracy_list,
        train_precision_list,
        train_recall_list,
        train_f1_list,

        accuracy_list,
        precision_list,
        recall_list,
        f1_list
    )),
    
    columns=[
        "Model Name",

        "Train Accuracy",
        "Train Precision",
        "Train Recall",
        "Train F1",

        "Test Accuracy",
        "Test Precision",
        "Test Recall",
        "Test F1"
    ]
)

results = results.sort_values(
    by="Test Accuracy",
    ascending=False
).reset_index(drop=True)

results

,Model Name,Train Accuracy,Train Precision,Train Recall,Train F1,Test Accuracy,Test Precision,Test Recall,Test F1
0,Hard Voting,1.000000,1.000000,1.000000,1.000000,0.889610,0.827586,0.872727,0.849558
1,Random Forest,1.000000,1.000000,1.000000,1.000000,0.883117,0.813559,0.872727,0.842105
2,Gradient Boosting,0.993485,1.000000,0.981221,0.990521,0.870130,0.807018,0.836364,0.821429
3,XGBoost,1.000000,1.000000,1.000000,1.000000,0.863636,0.814815,0.800000,0.807339
4,AdaBoost,0.915309,0.904523,0.845070,0.873786,0.857143,0.779661,0.836364,0.807018
5,Soft Voting,1.000000,1.000000,1.000000,1.000000,0.850649,0.775862,0.818182,0.796460
6,Decision Tree,1.000000,1.000000,1.000000,1.000000,0.844156,0.762712,0.818182,0.789474
7,SVM,0.887622,0.867347,0.798122,0.831296,0.824675,0.759259,0.745455,0.752294
8,KNN,0.864821,0.812500,0.793427,0.802850,0.798701,0.693548,0.781818,0.735043
9,Logistic Regression,0.773616,0.710227,0.586854,0.642674,0.772727,0.692308,0.654545,0.672897


In [15]:
## Now  I want to perform the Stacking on that using this 10 models 
from sklearn.ensemble import StackingClassifier


In [16]:
stacking_models = [
    ("rf", RandomForestClassifier(random_state=42)),
    ("svm", SVC()),
    ("xgb", XGBClassifier(random_state=42, eval_metric="logloss")),
    ("gb", GradientBoostingClassifier(random_state=42)),
    ("lr", LogisticRegression(max_iter=1000)),
    ("knn", KNeighborsClassifier()),
    ("nb", GaussianNB()),
    ("ada", AdaBoostClassifier(random_state=42)),
    ("dt", DecisionTreeClassifier(random_state=42)),

    ("hard_voting", VotingClassifier(
        estimators=[
            ("lr", LogisticRegression(max_iter=1000)),
            ("rf", RandomForestClassifier(random_state=42)),
            ("dt", DecisionTreeClassifier(random_state=42))
        ],
        voting="hard"
    )),

    ("soft_voting", VotingClassifier(
        estimators=[
            ("lr", LogisticRegression(max_iter=1000)),
            ("rf", RandomForestClassifier(random_state=42)),
            ("dt", DecisionTreeClassifier(random_state=42))
        ],
        voting="soft"
    ))
]

In [23]:
from sklearn.ensemble import StackingClassifier, RandomForestClassifier

stacking_model = StackingClassifier(
    estimators=stacking_models,
    final_estimator=RandomForestClassifier(
        n_estimators=100,
        random_state=42
    ),
    cv=5,
    stack_method="auto",
    n_jobs=-1
)

In [24]:
stacking_model.fit(X_train_scaled, y_train)

StackingClassifier(cv=5,
                   estimators=[('rf', RandomForestClassifier(random_state=42)),
                               ('svm', SVC()),
                               ('xgb',
                                XGBClassifier(base_score=None, booster=None,
                                              callbacks=None,
                                              colsample_bylevel=None,
                                              colsample_bynode=None,
                                              colsample_bytree=None,
                                              device=None,
                                              early_stopping_rounds=None,
                                              enable_categorical=False,
                                              eval_metric='logloss',
                                              feature_types=None, gamma=None,
                                              grow_...
                                                              RandomForestClassifier(random_state=42)),
                                                             ('dt',
                                                              DecisionTreeClassifier(random_state=42))])),
                               ('soft_voting',
                                VotingClassifier(estimators=[('lr',
                                                              LogisticRegression(max_iter=1000)),
                                                             ('rf',
                                                              RandomForestClassifier(random_state=42)),
                                                             ('dt',
                                                              DecisionTreeClassifier(random_state=42))],
                                                 voting='soft'))],
                   final_estimator=RandomForestClassifier(random_state=42),
                   n_jobs=-1)

In [25]:
y_train_pred = stacking_model.predict(X_train_scaled)
y_test_pred = stacking_model.predict(X_test_scaled)

In [26]:
train_accuracy, train_precision, train_recall, train_f1 = evaluate_model(
    y_train,
    y_train_pred
)

test_accuracy, test_precision, test_recall, test_f1 = evaluate_model(
    y_test,
    y_test_pred
)

In [27]:
print("STACKING CLASSIFIER - ALL 11 MODELS")
print("=" * 45)

print("Training Performance")
print("- Accuracy :", round(train_accuracy, 4))
print("- Precision:", round(train_precision, 4))
print("- Recall   :", round(train_recall, 4))
print("- F1 Score :", round(train_f1, 4))

print("-" * 45)

print("Testing Performance")
print("- Accuracy :", round(test_accuracy, 4))
print("- Precision:", round(test_precision, 4))
print("- Recall   :", round(test_recall, 4))
print("- F1 Score :", round(test_f1, 4))

STACKING CLASSIFIER - ALL 11 MODELS
Training Performance
- Accuracy : 0.9967
- Precision: 1.0
- Recall   : 0.9906
- F1 Score : 0.9953
---------------------------------------------
Testing Performance
- Accuracy : 0.8701
- Precision: 0.807
- Recall   : 0.8364
- F1 Score : 0.8214


In [28]:
stacking_result = pd.DataFrame({
    "Model Name": ["Stacking"],
    "Train Accuracy": [train_accuracy],
    "Train Precision": [train_precision],
    "Train Recall": [train_recall],
    "Train F1": [train_f1],
    "Test Accuracy": [test_accuracy],
    "Test Precision": [test_precision],
    "Test Recall": [test_recall],
    "Test F1": [test_f1]
})

results = pd.concat(
    [results, stacking_result],
    ignore_index=True
)

results = results.sort_values(
    by="Test Accuracy",
    ascending=False
).reset_index(drop=True)

results

,Model Name,Train Accuracy,Train Precision,Train Recall,Train F1,Test Accuracy,Test Precision,Test Recall,Test F1
0,Hard Voting,1.000000,1.000000,1.000000,1.000000,0.889610,0.827586,0.872727,0.849558
1,Random Forest,1.000000,1.000000,1.000000,1.000000,0.883117,0.813559,0.872727,0.842105
2,Gradient Boosting,0.993485,1.000000,0.981221,0.990521,0.870130,0.807018,0.836364,0.821429
3,Stacking,1.000000,1.000000,1.000000,1.000000,0.870130,0.807018,0.836364,0.821429
4,Stacking,0.996743,1.000000,0.990610,0.995283,0.870130,0.807018,0.836364,0.821429
5,XGBoost,1.000000,1.000000,1.000000,1.000000,0.863636,0.814815,0.800000,0.807339
6,AdaBoost,0.915309,0.904523,0.845070,0.873786,0.857143,0.779661,0.836364,0.807018
7,Soft Voting,1.000000,1.000000,1.000000,1.000000,0.850649,0.775862,0.818182,0.796460
8,Decision Tree,1.000000,1.000000,1.000000,1.000000,0.844156,0.762712,0.818182,0.789474
9,SVM,0.887622,0.867347,0.798122,0.831296,0.824675,0.759259,0.745455,0.752294


In [ ]:
'''from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(random_state=42)

param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [3, 5, 7, 10, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2"]
}

grid_search = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

grid_search.fit(X_train_scaled, y_train)

print("Best Parameters:")
print(grid_search.best_params_)

print("Best CV Accuracy:")
print(grid_search.best_score_)'''

Best Parameters:
{'max_depth': 10, 'max_features': 'log2', 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 300}
Best CV Accuracy:
0.8990803678528589


In [ ]:
'''best_rf = grid_search.best_estimator_

y_test_pred = best_rf.predict(X_test_scaled)

test_accuracy = accuracy_score(y_test, y_test_pred)
test_precision = precision_score(y_test, y_test_pred, zero_division=0)
test_recall = recall_score(y_test, y_test_pred, zero_division=0)
test_f1 = f1_score(y_test, y_test_pred, zero_division=0)

print("Tuned Random Forest")
print("- Accuracy :", round(test_accuracy, 4))
print("- Precision:", round(test_precision, 4))
print("- Recall   :", round(test_recall, 4))
print("- F1 Score :", round(test_f1, 4))'''

Tuned Random Forest
- Accuracy : 0.8701
- Precision: 0.7966
- Recall   : 0.8545
- F1 Score : 0.8246


In [31]:
from sklearn.ensemble import VotingClassifier

hard_voting = VotingClassifier(
    estimators=[
        ("lr", LogisticRegression(max_iter=1000)),
        ("rf", RandomForestClassifier(
            n_estimators=300,
            random_state=42
        )),
        ("dt", DecisionTreeClassifier(
            max_depth=5,
            random_state=42
        ))
    ],
    voting="hard"
)

In [32]:
hard_voting.fit(X_train_scaled, y_train)

VotingClassifier(estimators=[('lr', LogisticRegression(max_iter=1000)),
                             ('rf',
                              RandomForestClassifier(n_estimators=300,
                                                     random_state=42)),
                             ('dt',
                              DecisionTreeClassifier(max_depth=5,
                                                     random_state=42))])

In [33]:
y_test_pred = hard_voting.predict(X_test_scaled)

In [34]:
accuracy = accuracy_score(y_test, y_test_pred)
precision = precision_score(y_test, y_test_pred, zero_division=0)
recall = recall_score(y_test, y_test_pred, zero_division=0)
f1 = f1_score(y_test, y_test_pred, zero_division=0)

print("Hard Voting")
print("Accuracy :", round(accuracy, 4))
print("Precision:", round(precision, 4))
print("Recall   :", round(recall, 4))
print("F1 Score :", round(f1, 4))

Hard Voting
Accuracy : 0.8766
Precision: 0.8103
Recall   : 0.8545
F1 Score : 0.8319
